# Retail AI — YOLOv8 Training on SKU-110K (Kaggle)

**Before running:**
1. Go to `Settings → Accelerator → GPU P100`
2. Add the dataset: `Settings → Add Data → Search 'sku110k-annotations' by thedatasith → Add`
3. Run all cells top to bottom

**Advantages over Colab:**
- 30 hrs/week free GPU
- Dataset stays attached — no re-download each session
- Sessions last up to 9 hours
- Model saved to Kaggle Output — never lost

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# ── Cell 2: Install packages ─────────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
print('ultralytics ready')

In [ ]:
# ── Cell 3: Locate dataset ───────────────────────────────────────
# Dataset is already mounted — no download needed
import os
from pathlib import Path

# Kaggle mounts added datasets under /kaggle/input/<dataset-slug>/
INPUT_BASE = Path('/kaggle/input/sku110k-annotations')

# Find the SKU110K_fixed folder (pre-formatted YOLO layout)
DATASET_DIR = INPUT_BASE / 'SKU110K_fixed'

if not DATASET_DIR.exists():
    # Fallback: search for it
    import subprocess
    result = subprocess.run(['find', str(INPUT_BASE), '-type', 'd', '-name', 'SKU110K_fixed'],
                            capture_output=True, text=True)
    found = result.stdout.strip()
    if found:
        DATASET_DIR = Path(found.splitlines()[0])
    else:
        # Show full structure to debug
        print('SKU110K_fixed not found. Full structure:')
        for root, dirs, files in os.walk(INPUT_BASE):
            depth = str(root).replace(str(INPUT_BASE), '').count(os.sep)
            print('  ' * depth + os.path.basename(root) + '/')
            if depth < 2:
                for f in sorted(files)[:3]:
                    print('  ' * (depth+1) + f)
        raise FileNotFoundError('Could not locate SKU110K_fixed. Check dataset was added correctly.')

print(f'Dataset found: {DATASET_DIR}')
for split in ['train', 'val', 'test']:
    n_img = len(list((DATASET_DIR/'images'/split).glob('*.*')))
    n_lbl = len(list((DATASET_DIR/'labels'/split).glob('*.txt')))
    print(f'  {split}: {n_img} images, {n_lbl} labels')

In [ ]:
# ── Cell 4: Write data.yaml ──────────────────────────────────────
# Kaggle input is read-only, so write yaml to /kaggle/working/
import yaml
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(exist_ok=True)

yaml_content = {
    'path':  str(DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    1,
    'names': ['object']
}

yaml_path = WORK_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

print(f'data.yaml written → {yaml_path}')
print(yaml_content)

In [ ]:
# ── Cell 5: Train YOLOv8 on GPU ──────────────────────────────────
# Model options:
#   yolov8n.pt  nano   — fastest
#   yolov8s.pt  small  — recommended for Kaggle P100
#   yolov8m.pt  medium — best accuracy (slower)

from pathlib import Path
from ultralytics import YOLO

WORK_DIR   = Path('/kaggle/working')
yaml_path  = WORK_DIR / 'data.yaml'

MODEL  = 'yolov8s.pt'
EPOCHS = 50
IMGSZ  = 640
BATCH  = 16

model = YOLO(MODEL)
results = model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = 0,
    name    = 'sku_retail',
    project = str(WORK_DIR / 'runs'),
    patience= 10,
    plots   = True,
    workers = 2,
    cache   = True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.0, fliplr=0.5, mosaic=1.0,
    degrees=5.0, translate=0.1, scale=0.5,
)
print('Training complete!')
print(f'Best model: {WORK_DIR}/runs/sku_retail/weights/best.pt')

In [ ]:
# ── Cell 6: Evaluate and show metrics ────────────────────────────
from pathlib import Path
from ultralytics import YOLO

WORK_DIR  = Path('/kaggle/working')
yaml_path = WORK_DIR / 'data.yaml'
best      = WORK_DIR / 'runs' / 'sku_retail' / 'weights' / 'best.pt'

model   = YOLO(str(best))
metrics = model.val(data=str(yaml_path), split='test', plots=True)

box = metrics.box
f1  = 2 * box.mp * box.mr / (box.mp + box.mr + 1e-9)

print('\n=== EVALUATION RESULTS ===')
print(f'mAP@0.50       : {box.map50*100:.2f}%')
print(f'mAP@0.50:0.95  : {box.map*100:.2f}%')
print(f'Precision      : {box.mp*100:.2f}%')
print(f'Recall         : {box.mr*100:.2f}%')
print(f'F1 Score       : {f1*100:.2f}%')

In [ ]:
# ── Cell 7: Show training plots ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = Path('/kaggle/working/runs/sku_retail')
plots   = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, plot in zip(axes.flat, plots):
    p = run_dir / plot
    if p.exists():
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(plot.replace('.png', ''), fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/training_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved to /kaggle/working/training_summary.png')

In [ ]:
# ── Cell 8: Verify output files ──────────────────────────────────
# Everything in /kaggle/working/ is automatically saved as Kaggle Output.
# You can download best.pt directly from the Output tab on the right.
import os
from pathlib import Path

weights_dir = Path('/kaggle/working/runs/sku_retail/weights')
for f in weights_dir.glob('*.pt'):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'{f.name}  —  {size_mb:.1f} MB')

print('\nTo use in your app:')
print('1. Go to the Output tab (right panel) → Download best.pt')
print('2. Copy best.pt to retail-ai/backend/')
print('3. Update backend/.env:  YOLO_MODEL=best.pt')